In [1]:
import awkward as ak
import uproot
import numpy as np
import pandas as pd
import os
import json

from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

import sys
sys.path.append("..")
from src.processors.Processor import Processor

# Dataset

In [2]:
base = '/stash/user/fudawei/samples/mc/2018'
basedir = {d: os.path.join(base, d) for d in os.listdir(base)}
basedir

{'QCD': '/stash/user/fudawei/samples/mc/2018/QCD',
 'GJets': '/stash/user/fudawei/samples/mc/2018/GJets',
 'WJetsToQQ': '/stash/user/fudawei/samples/mc/2018/WJetsToQQ',
 'ZJetsToQQ': '/stash/user/fudawei/samples/mc/2018/ZJetsToQQ',
 'ZpToHGamma': '/stash/user/fudawei/samples/mc/2018/ZpToHGamma'}

In [3]:
samples = {s: [] for s in basedir}
for s in basedir:
    for (current_path, dirs, files) in os.walk(basedir[s]):
        for f in files:
            if f.endswith('.root'):
                samples[s].append(os.path.join(current_path, f))
samples

{'QCD': ['/stash/user/fudawei/samples/mc/2018/QCD/QCD_HT1500to2000_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/DC57E6C2-8124-5445-898B-1BBD937C06C1.root',
  '/stash/user/fudawei/samples/mc/2018/QCD/QCD_HT1500to2000_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/A93F7805-ED4E-BB48-9B3E-9DD930943A6A.root',
  '/stash/user/fudawei/samples/mc/2018/QCD/QCD_HT1500to2000_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/362B542E-D0F4-AE48-B225-2326228800B3.root',
  '/stash/user/fudawei/samples/mc/2018/QCD/QCD_HT1500to2000_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/8E6CA057-8342-224A-9681-4BC954F4CDFA.root',
  '/stash/user/fudawei/samples/mc/2018/QCD/QCD_HT1500to2000_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade

# Batch test

In [4]:
_events = NanoEventsFactory.from_root(file=samples['GJets'][0], treepath='Events', schemaclass=NanoAODSchema,).events()
p=Processor(outdir='../output/test/')
p.process(_events)

KeyboardInterrupt: 

# Parallel computing

In [7]:
cutflow = {}

In [9]:
cutflow['ZpToHGamma'] = processor.run_uproot_job(
    fileset={'ZpToHGamma': samples['ZpToHGamma']},
    treename="Events",
    processor_instance=Processor(outdir=os.path.join('../output/ZpToHGamma')),
    executor=processor.futures_executor,
    executor_args={"schema": NanoAODSchema, "workers": 12}, # running on $workers cpu cores
)

Output()

Output()

In [7]:
cutflow

{'ZpToHGamma': {'raw': 736000,
  'electron': 317369,
  'muon': 339720,
  'photon': 278180,
  'AK8jet': 184982,
  'trigger': 661355,
  'b-veto': 358691}}

In [10]:
cutflow['GJets'] = processor.run_uproot_job(
    fileset={'GJets': samples['GJets']},
    treename="Events",
    processor_instance=Processor(outdir=os.path.join('..', 'output', 'GJets')),
    executor=processor.futures_executor,
    executor_args={"schema": NanoAODSchema, "workers": 36}, # running on $workers cpu cores
)

Preprocessing:   0%|          | 0/137 [00:00<?, ?file/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:255: UserWarning: Cancelling 73 running jobs (likely due to an exception)
  warnings.warn(


OSError: expected Chunk of length 392,
received 0 bytes from MemmapSource
for file path /data/pubfs/fudawei/samples/mc/2018/GJets/nanov9/GJets_HT-600ToInf_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/F7D6B6E3-47C2-BD4D-A467-434C36B28B4F.root

In [7]:
cutflow['QCD'] = processor.run_uproot_job(
    fileset={'QCD': samples['QCD']},
    treename="Events",
    processor_instance=Processor(outdir=os.path.join('..', 'output', 'GJets')),
    executor=processor.futures_executor,
    executor_args={"schema": NanoAODSchema, "workers": 36}, # running on $workers cpu cores
)

Preprocessing:   0%|          | 0/704 [00:00<?, ?file/s]

/home/pku/fudawei/.local/lib/python3.8/site-packages/coffea/processor/executor.py:255: UserWarning: Cancelling 73 running jobs (likely due to an exception)
  warnings.warn(


OSError: expected Chunk of length 392,
received 0 bytes from MemmapSource
for file path /data/pubfs/fudawei/samples/mc/2018/QCD/nanov9/QCD_HT500to700_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/CCCEE5F3-3F4F-2145-B66F-F7F07879678C.root